In [ ]:
import os, sys

# Repository information
REPO_NAME = "RecSys-Challenge-2025"
REPO_URL  = f"github.com/Lv1g1/{REPO_NAME}.git"

# Detect environment
IS_COLAB = 'content' in os.getcwd()
IS_KAGGLE = 'kaggle' in os.getcwd()
IS_LOCAL = not (IS_COLAB or IS_KAGGLE)

if IS_COLAB:
    WORKING_DIR = "/content"

    # Mount Google Drive
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)

    # Get GitHub token via input
    def get_token():
        from getpass import getpass
        return getpass("GitHub Token: ")

elif IS_KAGGLE:
    WORKING_DIR = "/kaggle/working"

    # Get GitHub token from Kaggle secrets
    def get_token():
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("Token")

# If local environment assume inside the repo
LOCAL_REPO_PATH = os.getcwd() if IS_LOCAL else os.path.join(WORKING_DIR, REPO_NAME)

# Clone the repository if it doesn't exist
if not os.path.exists(LOCAL_REPO_PATH):
    os.chdir(WORKING_DIR)
    token = get_token()

    !git clone https://{token}@{REPO_URL}
else:
    print("Repo already exists — pulling latest changes")
    os.chdir(LOCAL_REPO_PATH)
    !git pull
    os.chdir(WORKING_DIR)

# Add to Python PATH
if LOCAL_REPO_PATH not in sys.path:
    sys.path.append(LOCAL_REPO_PATH)

In [ ]:
!pip install optuna
import optuna

In [ ]:
import importlib
import scipy.sparse as sps

from Challenge import paths
importlib.reload(paths)

from Evaluation.Evaluator import EvaluatorHoldout
from Challenge.hyper_tuning import hyperparameter_tuning

In [ ]:
# Load datasets
URM_train = sps.load_npz(paths.URM_TRAIN)
URM_validation = sps.load_npz(paths.URM_VALIDATION)

In [ ]:
# Set up evaluator
evaluator = EvaluatorHoldout(URM_validation, cutoff_list=[20])

In [ ]:
# Define objective function for hyperparameter tuning
from Recommenders.NonPersonalizedRecommender import GlobalEffects

STUDY_NAME = GlobalEffects.RECOMMENDER_NAME + "_hyperparameter_tuning"

def objective_function(optuna_trial: optuna.trial.Trial) -> float:
    recommender_instance = GlobalEffects(URM_train)
    recommender_instance.fit(
        # shrink factor
        lambda_user=optuna_trial.suggest_int("lambda_user", 0, 1000),
        lambda_item=optuna_trial.suggest_int("lambda_item", 0, 1000)
    )

    result_df, _ = evaluator.evaluateRecommender(recommender_instance)

    return result_df.loc[20]["RECALL"]

In [ ]:
# Perform hyperparameter tuning
save_results, optuna_study = hyperparameter_tuning(
    objective_function,
    study_name=STUDY_NAME,
    n_trials=50
)

In [ ]:
optuna.visualization.plot_optimization_history(optuna_study)

In [ ]:
optuna.visualization.plot_param_importances(optuna_study)

In [ ]:
optuna.visualization.plot_parallel_coordinate(optuna_study)

In [ ]:
# Train final model on train + validation with best hyperparameters
recommender = GlobalEffects(URM_train + URM_validation)
recommender.fit(
    lambda_user=optuna_study.best_trial.params["lambda_user"],
    lambda_item=optuna_study.best_trial.params["lambda_item"]
)

# Save the trained model
recommender.save_model(paths.MODEL_DIR)

In [ ]:
# Generate recommendations for the test set
URM_test = sps.load_npz(paths.URM_TEST)
evaluator_test = EvaluatorHoldout(URM_test, cutoff_list=[20])
recommendations = recommender.recommendAll(cutoff=20)
evaluator_test.save_recommendations(recommendations, paths.SUBMISSIONS)